In [1]:
import sys

sys.path.append("..")

import monte_carlo as mc
import model
import random
import yfinance as yf

### Setup parameters

- S: current stock price
- K: strike price

In [2]:
tkr = "AAPL"

t = yf.Ticker(tkr)
px = t.history(period="1d")["Close"].iloc[-1]
bump = 5

DAYS_TO_MATURE = 180
MODEL_PARAMS = dict(
    S=px, # stock price
    K=px + bump, # strike price
    T=(DAYS_TO_MATURE / 365), # time to maturity, converted to years
    r=0.05, # risk free rate
    sigma=1.586916, # volatility
    num_sims=100_000, # number of simulations
    days=DAYS_TO_MATURE, # number of days to simulate
    h=0.1, # increment size
    antithetic=True, # use variance reduction technique
)

### Price Options
Call and price using both available models, Black-Scholes and MonteCarlo

In [3]:
random.seed(42)

pricer_model = model.OptionsPricer(**MODEL_PARAMS)

# call to see all outputs
pricer_model.price_options()


*** BlackScholes Results ***
--------------------------------------------------
Call Option Price: 83.22
 Put Option Price: 83.3
       Delta Call: 0.7112
        Delta Put: -0.2888
            Gamma: 0.0016

 *** Monte Carlo Results ***
--------------------------------------------------
Call Option Price: 83.83 (+/- 1.7548)
 Put Option Price: 83.11  (+/- 0.4213)
       Delta call: 0.7152  (+/- 0.0105)
        Delta put: -0.2848  (+/- 0.0105)
            Gamma: 0.0014  (+/- 0.0006)
--------------------------------------------------


### Confidence Interval Calculation

In [4]:
mc_model = mc.MonteCarloModel(**MODEL_PARAMS)

print("\n *** Monte Carlo Results ***")
mc_model.run_simulation()

call_price = mc_model.call_option_price()
put_price = mc_model.put_option_price()
delta_call = mc_model.delta("call")
delta_put = mc_model.delta("put")
gamma = mc_model.gamma()

print("-" * 50)
print(f"Call Option Price: {round(call_price, 2)} (+/- {round(mc_model.ci_intervals['call'], 4)})")
print(f" Put Option Price: {round(put_price, 2)}  (+/- {round(mc_model.ci_intervals['put'], 4)})")
print(f"       Delta call: {round(delta_call, 4)}  (+/- {round(mc_model.ci_intervals['delta'], 4)})")
print(f"        Delta put: {round(delta_put, 4)}  (+/- {round(mc_model.ci_intervals['delta'], 4)})")
print(f"            Gamma: {round(gamma, 4)}  (+/- {round(mc_model.ci_intervals['gamma'], 4)})")
print("-" * 50)

mc_model.ci_intervals


 *** Monte Carlo Results ***
--------------------------------------------------
Call Option Price: 82.28 (+/- 1.7425)
 Put Option Price: 83.08  (+/- 0.4198)
       Delta call: 0.7056  (+/- 0.0104)
        Delta put: -0.2944  (+/- 0.0104)
            Gamma: 0.0015  (+/- 0.0006)
--------------------------------------------------


{'call': np.float64(1.7424671592935297),
 'put': np.float64(0.4197601566679349),
 'delta': np.float64(0.010440088376844127),
 'gamma': np.float64(0.0006011013229744138)}